# Resume Classification

This notebook develops a machine learning model to classify resumes
into their corresponding professional categories.

Workflow:

Resume Text -> TF-IDF Feature Extraction -> Machine Learning Classifier -> Predicted Resume Category -> Model Evaluation

In [1]:
import pandas as pd
import numpy as np

# Load cleaned resume dataset
resume_df = pd.read_csv("../data/processed/resumes_clean.csv")

print("Number of resumes:", len(resume_df))
print("Number of categories:", resume_df["Category"].nunique())

print("\nCategories:")
print(sorted(resume_df["Category"].unique()))

Number of resumes: 2484
Number of categories: 24

Categories:
['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


## Category Distribution

In [3]:
category_counts = resume_df["Category"].value_counts()

print(category_counts)

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


## Category Distribution Statistics

In [4]:
print("Largest category:", category_counts.index[0])
print("Largest category count:", category_counts.iloc[0])

print("\nSmallest category:", category_counts.index[-1])
print("Smallest category count:", category_counts.iloc[-1])

print("\nAverage resumes per category:",
      category_counts.mean())

Largest category: INFORMATION-TECHNOLOGY
Largest category count: 120

Smallest category: BPO
Smallest category count: 22

Average resumes per category: 103.5


## Train-Test Split

In [5]:
from sklearn.model_selection import train_test_split

X = resume_df["clean_resume"]
y = resume_df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training resumes:", len(X_train))
print("Testing resumes:", len(X_test))

print("\nTraining category distribution:")
print(y_train.value_counts())

print("\nTesting category distribution:")
print(y_test.value_counts())

Training resumes: 1987
Testing resumes: 497

Training category distribution:
Category
INFORMATION-TECHNOLOGY    96
BUSINESS-DEVELOPMENT      96
FINANCE                   94
ACCOUNTANT                94
FITNESS                   94
ADVOCATE                  94
CHEF                      94
ENGINEERING               94
SALES                     93
AVIATION                  93
BANKING                   92
HEALTHCARE                92
CONSULTANT                92
CONSTRUCTION              90
PUBLIC-RELATIONS          89
HR                        88
DESIGNER                  86
ARTS                      82
TEACHER                   82
APPAREL                   78
DIGITAL-MEDIA             77
AGRICULTURE               50
AUTOMOBILE                29
BPO                       18
Name: count, dtype: int64

Testing category distribution:
Category
ADVOCATE                  24
ENGINEERING               24
FINANCE                   24
INFORMATION-TECHNOLOGY    24
BUSINESS-DEVELOPMENT      24
CHEF  

In [7]:
print("Missing values in X_train:", X_train.isna().sum())
print("Missing values in X_test:", X_test.isna().sum())

Missing values in X_train: 1
Missing values in X_test: 0


In [8]:
print("Total missing clean resumes:", resume_df["clean_resume"].isna().sum())

Total missing clean resumes: 1


In [9]:
# Replace the single missing resume with an empty string
resume_df["clean_resume"] = resume_df["clean_resume"].fillna("")

# Recreate X and y after fixing the missing value
X = resume_df["clean_resume"]
y = resume_df["Category"]

# Perform the same stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Missing values in X_train:", X_train.isna().sum())
print("Missing values in X_test:", X_test.isna().sum())

Missing values in X_train: 0
Missing values in X_test: 0


## Term Frequency–Inverse Document Frequency.
Our machine-learning model cannot directly understand resume text. We first need to convert each resume into numerical features.
For this project, we'll use TF-IDF.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000, # Keeps at most 10,000 of the most useful text features. This prevents the feature space from becoming unnecessarily huge.
    stop_words="english", # Removes common words
    ngram_range=(1, 2), # It tells TF-IDF to learn both: Unigrams and Bigrams
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (1987, 10000)
Testing TF-IDF shape: (497, 10000)


## Train the Resume Classifier
Logistic Regression

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

model.fit(X_train_tfidf, y_train) # Train the classifier using the TF-IDF representation of the resumes and their known categories.

print("Model training completed.")

Model training completed.


## Make predictions on unseen resumes

In [15]:
y_pred = model.predict(X_test_tfidf)

print("Predictions generated:", len(y_pred))
print("\nFirst 10 predictions:")
print(y_pred[:10])

Predictions generated: 497

First 10 predictions:
['DESIGNER' 'DIGITAL-MEDIA' 'CONSTRUCTION' 'HEALTHCARE' 'CONSTRUCTION'
 'HR' 'ADVOCATE' 'DIGITAL-MEDIA' 'DIGITAL-MEDIA' 'DIGITAL-MEDIA']


## Evaluate the Classifier

In [17]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6559356136820925

Classification Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.65      0.83      0.73        24
              ADVOCATE       0.33      0.50      0.40        24
           AGRICULTURE       1.00      0.38      0.56        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.50      0.29      0.36        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.85      0.71      0.77        24
               BANKING       0.79      0.65      0.71        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.54      0.92      0.68        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.77      0.77      0.77        22
            CONSULTANT       0.50      0.22      0.30        23
              DESIGNER       0.86      0.86      0

C:\my work\SkillSync AI\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\my work\SkillSync AI\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\my work\SkillSync AI\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Confusion Matrix

In [18]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=model.classes_
)

cm_df = pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_
)

print(cm_df)

                        ACCOUNTANT  ADVOCATE  AGRICULTURE  APPAREL  ARTS  \
ACCOUNTANT                      20         2            0        0     0   
ADVOCATE                         0        12            0        0     1   
AGRICULTURE                      0         2            5        0     1   
APPAREL                          0         2            0        4     0   
ARTS                             0         0            0        0     6   
AUTOMOBILE                       0         5            0        0     0   
AVIATION                         0         0            0        0     0   
BANKING                          2         2            0        0     0   
BPO                              0         0            0        0     0   
BUSINESS-DEVELOPMENT             1         0            0        0     0   
CHEF                             0         2            0        0     0   
CONSTRUCTION                     0         1            0        0     0   
CONSULTANT  

In [19]:
for category in model.classes_:
    actual = cm_df.loc[category]
    mistakes = actual[actual.index != category].sort_values(ascending=False)

    print(f"\n{category}")
    print(mistakes.head(3))


ACCOUNTANT
ADVOCATE       2
SALES          2
AGRICULTURE    0
Name: ACCOUNTANT, dtype: int64

ADVOCATE
HEALTHCARE    6
SALES         2
ARTS          1
Name: ADVOCATE, dtype: int64

AGRICULTURE
ADVOCATE            2
PUBLIC-RELATIONS    2
ARTS                1
Name: AGRICULTURE, dtype: int64

APPAREL
SALES                   5
BUSINESS-DEVELOPMENT    4
ADVOCATE                2
Name: APPAREL, dtype: int64

ARTS
TEACHER                   7
INFORMATION-TECHNOLOGY    2
AVIATION                  1
Name: ARTS, dtype: int64

AUTOMOBILE
ADVOCATE       5
AVIATION       1
ENGINEERING    1
Name: AUTOMOBILE, dtype: int64

AVIATION
ENGINEERING             3
BUSINESS-DEVELOPMENT    1
HR                      1
Name: AVIATION, dtype: int64

BANKING
ACCOUNTANT    2
ADVOCATE      2
CONSULTANT    1
Name: BANKING, dtype: int64

BPO
CONSULTANT              1
BUSINESS-DEVELOPMENT    1
BANKING                 1
Name: BPO, dtype: int64

BUSINESS-DEVELOPMENT
ACCOUNTANT    1
HEALTHCARE    1
ADVOCATE      0
Name:

## Check the model's confidence

In [20]:
# Get prediction probabilities
y_proba = model.predict_proba(X_test_tfidf)

# Highest probability for each prediction
max_probabilities = y_proba.max(axis=1)

print("Average prediction confidence:", max_probabilities.mean())
print("Minimum prediction confidence:", max_probabilities.min())
print("Maximum prediction confidence:", max_probabilities.max())

Average prediction confidence: 0.35560054744483055
Minimum prediction confidence: 0.0686543414832634
Maximum prediction confidence: 0.9270354585103973


In [21]:
import pandas as pd

confidence_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Confidence": max_probabilities
})

confidence_df = confidence_df.sort_values(
    "Confidence",
    ascending=True
)

print(confidence_df.head(20))

               Actual         Predicted  Confidence
154     DIGITAL-MEDIA              ARTS    0.068654
255       AGRICULTURE       ENGINEERING    0.079160
485           BANKING  PUBLIC-RELATIONS    0.079708
464           FITNESS           FITNESS    0.080061
448              CHEF           FINANCE    0.081474
463           APPAREL          ADVOCATE    0.081623
445        ACCOUNTANT          ADVOCATE    0.081821
302  PUBLIC-RELATIONS        HEALTHCARE    0.085272
358        CONSULTANT        CONSULTANT    0.085276
407          ADVOCATE          ADVOCATE    0.085880
344          AVIATION          AVIATION    0.085930
313           FINANCE          ADVOCATE    0.086536
349              CHEF          ADVOCATE    0.086820
37        AGRICULTURE  PUBLIC-RELATIONS    0.087198
61   PUBLIC-RELATIONS          ADVOCATE    0.087604
129           APPAREL          DESIGNER    0.087853
317           FINANCE           FINANCE    0.089290
132           TEACHER           TEACHER    0.089932
172         

## Improve the classifier 

In [22]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

print("Linear SVM training completed.")

Linear SVM training completed.


In [23]:
y_pred_svm = svm_model.predict(X_test_tfidf)

print("Predictions generated:", len(y_pred_svm))

Predictions generated: 497


In [24]:
from sklearn.metrics import accuracy_score, classification_report

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Accuracy:", svm_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    zero_division=0
))

Linear SVM Accuracy: 0.7323943661971831

Classification Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.72      0.88      0.79        24
              ADVOCATE       0.57      0.71      0.63        24
           AGRICULTURE       0.89      0.62      0.73        13
               APPAREL       0.58      0.37      0.45        19
                  ARTS       0.62      0.38      0.47        21
            AUTOMOBILE       1.00      0.43      0.60         7
              AVIATION       0.82      0.75      0.78        24
               BANKING       0.78      0.78      0.78        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.70      0.88      0.78        24
                  CHEF       0.86      0.79      0.83        24
          CONSTRUCTION       0.76      0.86      0.81        22
            CONSULTANT       0.77      0.43      0.56        23
              DESIGNER       0.91      

### We established a Logistic Regression baseline and then compared it with a Linear SVM using the same TF-IDF representation and the same stratified train/test split.

In [28]:
from sklearn.metrics import confusion_matrix

cm_svm = confusion_matrix(
    y_test,
    y_pred_svm,
    labels=svm_model.classes_
)

cm_svm_df = pd.DataFrame(
    cm_svm,
    index=svm_model.classes_,
    columns=svm_model.classes_
)

print(cm_svm_df)

                        ACCOUNTANT  ADVOCATE  AGRICULTURE  APPAREL  ARTS  \
ACCOUNTANT                      21         1            0        0     0   
ADVOCATE                         0        17            0        1     1   
AGRICULTURE                      0         1            8        0     1   
APPAREL                          0         2            0        7     1   
ARTS                             0         1            0        0     8   
AUTOMOBILE                       0         2            0        0     0   
AVIATION                         0         0            0        0     0   
BANKING                          1         0            0        0     0   
BPO                              0         0            0        0     0   
BUSINESS-DEVELOPMENT             1         0            0        0     1   
CHEF                             0         1            0        0     0   
CONSTRUCTION                     0         0            0        0     0   
CONSULTANT  

In [29]:
for category in svm_model.classes_:
    actual = cm_svm_df.loc[category]
    mistakes = actual[actual.index != category].sort_values(ascending=False)

    print(f"\n{category}")
    print(mistakes.head(3))


ACCOUNTANT
SALES          2
ADVOCATE       1
AGRICULTURE    0
Name: ACCOUNTANT, dtype: int64

ADVOCATE
HEALTHCARE    4
ARTS          1
APPAREL       1
Name: ADVOCATE, dtype: int64

AGRICULTURE
ADVOCATE        1
ARTS            1
CONSTRUCTION    1
Name: AGRICULTURE, dtype: int64

APPAREL
ADVOCATE    2
SALES       2
ARTS        1
Name: APPAREL, dtype: int64

ARTS
TEACHER     6
AVIATION    1
ADVOCATE    1
Name: ARTS, dtype: int64

AUTOMOBILE
ADVOCATE       2
AVIATION       1
ENGINEERING    1
Name: AUTOMOBILE, dtype: int64

AVIATION
ENGINEERING             3
BUSINESS-DEVELOPMENT    1
HR                      1
Name: AVIATION, dtype: int64

BANKING
ACCOUNTANT          1
PUBLIC-RELATIONS    1
CONSULTANT          1
Name: BANKING, dtype: int64

BPO
HEALTHCARE    1
SALES         1
BANKING       1
Name: BPO, dtype: int64

BUSINESS-DEVELOPMENT
ACCOUNTANT    1
ARTS          1
HR            1
Name: BUSINESS-DEVELOPMENT, dtype: int64

CHEF
ADVOCATE        1
AVIATION        1
CONSTRUCTION    1
Name: 

## Save the trained model

1. The trained Linear SVM
2. The trained TF-IDF vectorizer

If we only save the SVM and not the vectorizer, we wouldn't know how to convert new resume text into the same feature representation the model learned during training.

In [31]:
import joblib
import os

# Make sure the models folder exists
os.makedirs("../models", exist_ok=True)

# Save the trained SVM classifier
joblib.dump(
    svm_model,
    "../models/resume_classifier.pkl"
)

# Save the TF-IDF vectorizer
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

print("Model saved successfully.")
print("TF-IDF vectorizer saved successfully.")

Model saved successfully.
TF-IDF vectorizer saved successfully.


In [32]:
loaded_model = joblib.load("../models/resume_classifier.pkl")
loaded_tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

print("Saved model loaded successfully.")
print("Saved vectorizer loaded successfully.")

# Test that they still produce predictions
test_features = loaded_tfidf.transform(X_test)

loaded_predictions = loaded_model.predict(test_features)

print("Loaded model predictions:", len(loaded_predictions))
print("Predictions match original:",
      np.array_equal(y_pred_svm, loaded_predictions))

Saved model loaded successfully.
Saved vectorizer loaded successfully.
Loaded model predictions: 497
Predictions match original: True


## Test the model on a NEW resume

In [35]:
import re

def clean_resume_text(text):
    text = str(text)
    
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    
    text = text.strip()
    
    return text

In [36]:
def predict_resume_category(resume_text):
    resume_text = str(resume_text)

    # Apply the same cleaning used during training
    resume_text = clean_resume_text(resume_text)

    # Convert the resume into TF-IDF features
    resume_features = loaded_tfidf.transform([resume_text])

    # Predict category
    prediction = loaded_model.predict(resume_features)[0]

    return prediction

In [37]:
new_resume = """
Software Engineer with experience in Python, Java, SQL, React.js and Node.js.
Experienced in developing web applications and REST APIs.
Worked with MongoDB, PostgreSQL, Git and Docker.
Strong knowledge of machine learning, data analysis and software development.
"""

predicted_category = predict_resume_category(new_resume)

print("Predicted Resume Category:", predicted_category)

Predicted Resume Category: ENGINEERING
